In [123]:
## Load my files ##
import sys
sys.path.append('..')
from utils import get_sequence

## Load standard files ##
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torch.optim.lr_scheduler as lr_scheduler
from torch import from_numpy as tnsr
from scipy.stats import bernoulli
import torch.nn as nn
import numpy as np
import pandas as pd
from tqdm import tqdm
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.spatial.distance import cdist as dist
from sklearn.metrics.pairwise import cosine_similarity
from scipy.signal import find_peaks
from scipy.spatial.distance import cosine
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
import torch.nn.functional as F
import random 

In [124]:
def fix_seeds(seed=42):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)  # if using multi-GPU
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [125]:
n_community = 2
n_members = 3

tokens = []

for ii in range(n_community*n_members+1):
    tokens.append(
        chr(ord('A')+ii)
    )

In [126]:
class RNN1(nn.Module):
    def __init__(self, input_size, hidden_size, hidden_sleep_size, context_size, output_size=7, num_layers=1, modulation_strength=0):
        super(RNN1, self).__init__()

        self.context_fc = nn.Linear(hidden_sleep_size, context_size, bias=False)
        self.rnn = nn.RNN(input_size, hidden_size, num_layers, nonlinearity='relu', batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)
        self.modulation_strength = modulation_strength

    def set_modulation(self, val):
        self.modulation_strength = val
        
    def forward(self, x, hs, h=None):
        context = self.context_fc(hs)
        # print(hs, context, 'sjs')
        x = torch.cat(((1-self.modulation_strength)*x, self.modulation_strength*context), dim=2)
        # print(x)
        out, h = self.rnn(x, h)
        out = self.fc(out[:,-1,:])

        return out, h

In [127]:
class RNN2(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, num_layers=1):
        super(RNN2, self).__init__()

        self.rnn = nn.RNN(input_size, hidden_size, num_layers, nonlinearity='relu', batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)

        self._init_weights_to_zero()

    def forward(self, x, h=None):
        out, h = self.rnn(x, h)
        out = self.fc(out[:,-1,:])

        return out, h
    
    def _init_weights_to_zero(self):
        # Zero out RNN weights and biases
        for name, param in self.rnn.named_parameters():
            nn.init.constant_(param, 0.0)

        # Zero out Linear layer weights and biases
        nn.init.constant_(self.fc.weight, 0.0)
        nn.init.constant_(self.fc.bias, 0.0)

In [128]:
class Dataset_converter(Dataset):
    def __init__(self, data, working_memory=1, short_term_memory=8):
        
        one_hot_encoded = np.zeros((len(data), len(tokens)), dtype=float)
        for ii, token in enumerate(data):
            one_hot_encoded[ii,ord(token)-65] = 1
        
        self.X = np.zeros((((len(data)-working_memory-short_term_memory)), short_term_memory, len(tokens)*working_memory))
        self.y = np.zeros((((len(data)-working_memory-short_term_memory)), len(tokens)))

        for ii in range(self.X.shape[0]):
            for jj in range(self.X.shape[1]):
                for kk in range(working_memory):
                    self.X[ii,jj,kk*len(tokens):(kk+1)*len(tokens)] = \
                    one_hot_encoded[ii+jj+kk,:]
                    
            self.y[ii] = \
                one_hot_encoded[ii+jj+kk+1,:]

        self.X = tnsr(self.X).float()
        self.y = tnsr(self.y).float()

    def __getitem__(self, index):
        return self.X[index], self.y[index]

    def __len__(self):
        return self.X.shape[0]

In [129]:
### initial training ###
total_samples = 20000
working_memory = 1
short_term_memory = 1
hidden_wake_size = 40
hidden_sleep_size = 100
context_output_size = 10
num_layers_wake = 1
num_layers_sleep = 1
output_size = len(tokens)
input_size = len(tokens)*working_memory
lr = 1e-3
test_acc = []
c = []
cortical_strength = .5

data = get_sequence(total_samples, n_community, n_members, train_percent=1.0)#gen_seq(total_samples) #

data_set = Dataset_converter(data, working_memory, short_term_memory)
train_loader = DataLoader(data_set, batch_size=1, shuffle=False)

network1 = RNN1(input_size+context_output_size, hidden_wake_size, hidden_sleep_size, context_size=context_output_size, output_size=output_size, num_layers=num_layers_wake)
network1.set_modulation(cortical_strength)

optimizer = torch.optim.SGD(network1.parameters(), lr=lr, momentum=0.95)
criterion = torch.nn.CrossEntropyLoss()

total = 0
correct = np.zeros(1000,dtype=float)
hs = torch.zeros((1,1,hidden_sleep_size))
for X, y in train_loader:
    # context = torch.zeros((1,X.size(1),context_output_size))
    # X = torch.cat((X,context), dim=2)
    optimizer.zero_grad()

    if total == 0:
        predicted_y, h = network1(X,hs)
    else:
        predicted_y, h = network1(X, hs, h=mem)
    
    loss = criterion(predicted_y, y)
    
    
    loss.backward(retain_graph=True)
    optimizer.step()

    with torch.no_grad():
        mem = h.clone()
        true_y = y.argmax(axis=1)
        estimated_y = predicted_y.argmax(axis=1)

        total += 1
        if true_y == estimated_y:
                correct[total%1000] = 1
        else:
            correct[total%1000] = 0

        test_acc.append(
            np.sum(correct)/total if total<1000 else np.sum(correct)/1000
        )
        if total%1000 == 0:
            print(f'Iter : {total+1}, loss: {loss:.4f}, accuracy: {test_acc[-1]:.4f}')


Iter : 1001, loss: 2.0833, accuracy: 0.2420
Iter : 2001, loss: 1.9529, accuracy: 0.2500
Iter : 3001, loss: 1.6347, accuracy: 0.3540
Iter : 4001, loss: 2.9486, accuracy: 0.5290
Iter : 5001, loss: 1.0828, accuracy: 0.5530
Iter : 6001, loss: 2.2134, accuracy: 0.6220
Iter : 7001, loss: 1.6716, accuracy: 0.6300
Iter : 8001, loss: 2.1612, accuracy: 0.6370
Iter : 9001, loss: 2.4972, accuracy: 0.6620
Iter : 10001, loss: 1.9800, accuracy: 0.6600
Iter : 11001, loss: 1.7289, accuracy: 0.6360
Iter : 12001, loss: 1.8641, accuracy: 0.6650
Iter : 13001, loss: 2.7180, accuracy: 0.6680
Iter : 14001, loss: 1.8711, accuracy: 0.6560
Iter : 15001, loss: 1.8251, accuracy: 0.6530
Iter : 16001, loss: 1.8070, accuracy: 0.6670
Iter : 17001, loss: 1.6074, accuracy: 0.6640
Iter : 18001, loss: 1.8853, accuracy: 0.6600
Iter : 19001, loss: 1.6755, accuracy: 0.6670


# Test randomness with noise injection

In [130]:
total_samples = 1000
idx = torch.randint(0, len(tokens), (1,)) [0]
X_hat = torch.zeros(len(tokens),dtype=torch.float32)
X_hat[idx] = 1.0
cortical_strength = .01
SWR = 0
counts = []
seq = ''

network1.set_modulation(cortical_strength)
hs = torch.randn((1,1,hidden_sleep_size))
network2 = RNN2(hidden_wake_size, hidden_sleep_size, hidden_wake_size, num_layers=num_layers_sleep)
optimizer = torch.optim.SGD(network2.parameters(), lr=lr, momentum=0.95)
criterion = torch.nn.MSELoss()

prev_h = torch.randn((1,1,hidden_wake_size))
mem = hs.clone()
for jj in range(total_samples):
    fix_seeds()
    with torch.no_grad():
        if jj == 0:
            X_hat_, hidden_state = network1(X_hat.reshape(1,1,-1), hs)
        else:
            hidden_state += SWR*torch.randn(hidden_state.size())
            X_hat_, hidden_state = network1(X_hat.reshape(1,1,-1), hs, hidden_state)

        h_target = hidden_state.clone()

    ### train RNN2 ###
    optimizer.zero_grad()
    predicted_h, hs = network2(prev_h, mem)
    loss = criterion(predicted_h[0][0], h_target[0][0])
    
    
    loss.backward(retain_graph=True)
    optimizer.step()

    with torch.no_grad():
        mem = hs.clone()
        X_hat_prob = torch.nn.functional.softmax(X_hat_, dim=1)
        fix_seeds(10)
        dist_categ = torch.distributions.Categorical(probs=X_hat_prob.reshape(-1))
        idx = dist_categ.sample()
        # idx = X_hat_prob.argmax()

        # noise = 10*torch.randn(context_output_size)
        X_hat = torch.zeros(len(tokens),dtype=torch.float32)
        # X_hat[len(tokens):] = noise
        X_hat[idx] = 1.0
        X_hat = X_hat.reshape(1,1,-1)
        seq = seq + tokens[idx]
        prev_h = hidden_state.clone()

        if jj%10000==0:
            print(loss)

        

tensor(0.3234, grad_fn=<MseLossBackward0>)


In [131]:
seq[:100]

'DFGGCABGFDEGFDEGFDEGFDEGFDEGFDEGFDEGFDEGFDEGFDEGFDEGFDEGFDEGFDEGFDEGFDEGFDEGFDEGFDEGFDEGFDEGFDEGFDEG'

In [132]:
total_samples = 1000
idx = torch.randint(0, len(tokens), (1,)) [0]
X_hat = torch.zeros(len(tokens),dtype=torch.float32)
X_hat[idx] = 1.0
cortical_strength = .01
SWR = .7
counts = []
seq_ = ''

network1.set_modulation(cortical_strength)
hs = torch.randn((1,1,hidden_sleep_size))
network2 = RNN2(hidden_wake_size, hidden_sleep_size, hidden_wake_size, num_layers=num_layers_sleep)
optimizer = torch.optim.SGD(network2.parameters(), lr=lr, momentum=0.95)
criterion = torch.nn.MSELoss()

prev_h = torch.randn((1,1,hidden_wake_size))
mem = hs.clone()
for jj in range(total_samples):
    fix_seeds()
    with torch.no_grad():
        if jj == 0:
            X_hat_, hidden_state = network1(X_hat.reshape(1,1,-1), hs)
        else:
            hidden_state += SWR*torch.randn(hidden_state.size())
            X_hat_, hidden_state = network1(X_hat.reshape(1,1,-1), hs, hidden_state)

        h_target = hidden_state.clone()

    ### train RNN2 ###
    optimizer.zero_grad()
    predicted_h, hs = network2(prev_h, mem)
    loss = criterion(predicted_h[0][0], h_target[0][0])
    
    
    loss.backward(retain_graph=True)
    optimizer.step()

    with torch.no_grad():
        mem = hs.clone()
        X_hat_prob = torch.nn.functional.softmax(X_hat_, dim=1)
        fix_seeds(10)
        dist_categ = torch.distributions.Categorical(probs=X_hat_prob.reshape(-1))
        idx = dist_categ.sample()
        # idx = X_hat_prob.argmax()

        # noise = 10*torch.randn(context_output_size)
        X_hat = torch.zeros(len(tokens),dtype=torch.float32)
        # X_hat[len(tokens):] = noise
        X_hat[idx] = 1.0
        X_hat = X_hat.reshape(1,1,-1)
        seq_ = seq_ + tokens[idx]
        prev_h = hidden_state.clone()

        if jj%10000==0:
            print(loss)

        

tensor(0.2998, grad_fn=<MseLossBackward0>)


In [133]:
seq[:100]

'DFGGCABGFDEGFDEGFDEGFDEGFDEGFDEGFDEGFDEGFDEGFDEGFDEGFDEGFDEGFDEGFDEGFDEGFDEGFDEGFDEGFDEGFDEGFDEGFDEG'

In [134]:
seq_[:100]

'BGCGFGDGFGDGFGDGFGDGFGDGFGDGFGDGFGDGFGDGFGDGFGDGFGDGFGDGFGDGFGDGFGDGFGDGFGDGFGDGFGDGFGDGFGDGFGDGFGDG'

# Do iterative sleep and wake stages

In [147]:
class brain:
    def __init__(self, input_size, hidden_wake_size, hidden_sleep_size, context_output_size, num_layers_wake, num_layers_sleep, output_size=7, wake_lr=1e-3, sleep_lr=1e-3):
        self.wake_model = RNN1(input_size+context_output_size, hidden_wake_size, hidden_sleep_size, context_size=context_output_size, output_size=output_size, num_layers=num_layers_wake)
        self.wake_optimizer = torch.optim.SGD(self.wake_model.parameters(), lr=wake_lr, momentum=0.95)
        self.wake_criterion = torch.nn.CrossEntropyLoss()

        self.sleep_model = RNN2(hidden_wake_size, hidden_sleep_size, hidden_wake_size, num_layers=num_layers_sleep)
        self.sleep_optimizer = torch.optim.SGD(self.sleep_model.parameters(), lr=sleep_lr, momentum=0.95)
        self.sleep_criterion = torch.nn.MSELoss()

        self.hidden_wake_size = hidden_wake_size
        self.hidden_sleep_size = hidden_sleep_size

    def wake(self, train_loader, cortical_strength=0.5):
        self.wake_model.set_modulation(cortical_strength)
        total = 0
        test_acc = []
        correct = np.zeros(1000,dtype=float)
        hs = torch.zeros((1,1,self.hidden_sleep_size))
        for X, y in train_loader:
            self.wake_optimizer.zero_grad()

            if total == 0:
                predicted_y, h = self.wake_model(X,hs)
            else:
                predicted_y, h = self.wake_model(X, hs, h=mem)
            
            loss = self.wake_criterion(predicted_y, y)
            
            
            loss.backward(retain_graph=True)
            self.wake_optimizer.step()

            with torch.no_grad():
                mem = h.clone()
                _, hs = self.sleep_model(h, hs)

                true_y = y.argmax(axis=1)
                estimated_y = predicted_y.argmax(axis=1)

                total += 1
                if true_y == estimated_y:
                        correct[total%1000] = 1
                else:
                    correct[total%1000] = 0

                test_acc.append(
                    np.sum(correct)/total if total<1000 else np.sum(correct)/1000
                )
                if total%1000 == 0:
                    print(f'Iter : {total+1}, loss: {loss:.4f}, accuracy: {test_acc[-1]:.4f}')
            
        return test_acc
    
    def sleep(self, sleep_duration=1000000, cortical_strength=0.01, SWR=1.1):
        self.wake_model.set_modulation(cortical_strength)
        prev_h = torch.randn((1,1,self.hidden_wake_size))
        hs = torch.randn((1,1,self.hidden_sleep_size))
        mem = hs.clone()
        idx = torch.randint(0, len(tokens), (1,)) [0]
        X_hat = torch.zeros(len(tokens),dtype=torch.float32)
        X_hat[idx] = 1.0
        for jj in range(sleep_duration):
            with torch.no_grad():
                if jj == 0:
                    X_hat_, hidden_state = self.wake_model(X_hat.reshape(1,1,-1), hs)
                else:
                    hidden_state += SWR*torch.randn(hidden_state.size())
                    X_hat_, hidden_state = self.wake_model(X_hat.reshape(1,1,-1), hs, hidden_state)

                h_target = hidden_state.clone()

            ### train sleep model ###
            self.sleep_optimizer.zero_grad()
            predicted_h, hs = self.sleep_model(prev_h, mem)
            # print(predicted_h[0], h_target[0])
            loss = self.sleep_criterion(predicted_h[0], h_target[0][0])
            
            
            loss.backward(retain_graph=True)
            optimizer.step()

            with torch.no_grad():
                mem = hs.clone()
                X_hat_prob = torch.nn.functional.softmax(X_hat_, dim=1)
                dist_categ = torch.distributions.Categorical(probs=X_hat_prob.reshape(-1))
                idx = dist_categ.sample()
                
                X_hat = torch.zeros(len(tokens),dtype=torch.float32)
                X_hat[idx] = 1.0
                X_hat = X_hat.reshape(1,1,-1)
                prev_h = hidden_state.clone()

                if jj%10000==0:
                    print('Sleep loss iteration ',jj,': ',loss)

In [148]:
# total_samples = 200000
working_memory = 1
short_term_memory = 1
hidden_wake_size = 40
hidden_sleep_size = 100
context_output_size = 10
num_layers_wake = 1
num_layers_sleep = 1
output_size = len(tokens)
input_size = len(tokens)*working_memory
lr = 1e-3
test_acc = []
wake_sleep_cycle = 3


model = brain(input_size, hidden_wake_size, hidden_sleep_size, context_output_size, num_layers_wake, num_layers_sleep, output_size=output_size, wake_lr=lr, sleep_lr=lr)

for cycle in range(wake_sleep_cycle):
    if cycle == 0:
        total_samples = 30000
    else:
        total_samples = 100000

    data = get_sequence(total_samples, n_community, n_members, train_percent=1.0)

    data_set = Dataset_converter(data, working_memory, short_term_memory)
    train_loader = DataLoader(data_set, batch_size=1, shuffle=False)

    model.wake(train_loader)
    model.sleep()

Iter : 1001, loss: 2.0279, accuracy: 0.2390
Iter : 2001, loss: 1.9948, accuracy: 0.2670
Iter : 3001, loss: 1.4963, accuracy: 0.4360
Iter : 4001, loss: 3.3810, accuracy: 0.5360
Iter : 5001, loss: 2.2081, accuracy: 0.5530
Iter : 6001, loss: 1.8891, accuracy: 0.5850
Iter : 7001, loss: 2.6152, accuracy: 0.6560
Iter : 8001, loss: 1.8064, accuracy: 0.6380
Iter : 9001, loss: 1.9500, accuracy: 0.6570
Iter : 10001, loss: 1.7603, accuracy: 0.6520
Iter : 11001, loss: 1.7773, accuracy: 0.6670
Iter : 12001, loss: 1.6664, accuracy: 0.6530
Iter : 13001, loss: 1.6336, accuracy: 0.6600
Iter : 14001, loss: 1.5696, accuracy: 0.6750
Iter : 15001, loss: 2.0379, accuracy: 0.6530
Iter : 16001, loss: 1.7698, accuracy: 0.6780
Iter : 17001, loss: 1.8958, accuracy: 0.6640
Iter : 18001, loss: 1.9971, accuracy: 0.6790
Iter : 19001, loss: 1.7289, accuracy: 0.6640
Iter : 20001, loss: 1.9236, accuracy: 0.6530
Iter : 21001, loss: 1.4976, accuracy: 0.6760
Iter : 22001, loss: 1.7163, accuracy: 0.6590
Iter : 23001, loss:

In [105]:
hs

tensor([[[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0., 0., 0., 0., 0., 0.]]], grad_fn=<StackBackward0>)